# Clinical Analytics Chatbot
**Drug R&D Assistant — Dataiku Notebook UI**

Run all cells in order. The interactive chat UI launches in the last cell (Cell 4).

**Before running:** set your LLM Mesh connection ID in Cell 2.

`lib/python/` is automatically on the Python path in all Dataiku notebooks — no path manipulation needed.

In [ ]:
# Cell 1 — Initialise Panel
import panel as pn
pn.extension('bokeh', sizing_mode='stretch_width')

In [ ]:
# Cell 2 — Configuration
# Replace with your actual Dataiku LLM Mesh connection ID
LLM_CONNECTION_ID = "YOUR_LLM_CONNECTION_ID"

In [ ]:
# Cell 3 — Initialise backend
import sys

# Flush cached backend modules so updated code is always loaded fresh
for _mod in list(sys.modules.keys()):
    if _mod.startswith('backend'):
        del sys.modules[_mod]

import yaml
from pathlib import Path
import backend

config_dir = Path(backend.__file__).parent.parent / "config"
with open(config_dir / "llm_config.yaml") as f:
    config = yaml.safe_load(f)
config["llm_mesh"]["connection_id"] = LLM_CONNECTION_ID

from backend.state.session_store import SessionStore
from backend.orchestrator.orchestrator import Orchestrator

session_store = SessionStore(timeout_minutes=60)
orchestrator = Orchestrator(session_store, config=config)

# ── Monkey-patch LLMClient.complete (Dataiku workdir cache bypass) ────────────
from backend.llm.llm_client import LLMClient

def _patched_complete(self, messages, temperature=None):
    if not hasattr(self, "call_log"):
        self.call_log = []
    try:
        api_client = self._get_dataiku_client()
        project_key = __import__("dataiku").default_project_key()
        project = api_client.get_project(project_key)
        llm = project.get_llm(self.connection_id)
        completion = llm.new_completion()
        for msg in messages:
            completion.with_message(msg["content"], msg["role"])
        resp = completion.execute()
        self.call_log.append({"messages": messages, "response": resp.text})
        return resp.text
    except Exception as e:
        import logging
        logging.getLogger(__name__).error("LLM Mesh call failed: %s", e)
        self.call_log.append({"messages": messages, "response": f"ERROR: {e}", "error": True})
        raise

LLMClient.complete = _patched_complete
orchestrator.llm.call_log = []



# ── Patch for site_list_matching (workdir-independent) ───────────────────────
# The Dataiku workdir may have stale Python/YAML files referencing site_list_merger.
# We define the matching agent inline and alias both skill IDs so either old or new
# workdir code paths work without error.

import os as _os
import pandas as _pd
from pathlib import Path as _Path

# -- Inline prompt templates (workdir may have old versions) ------------------
_SITE_MATCHING_SYSTEM = """You are an expert clinical operations data specialist.
Your task is to semantically match an uploaded list of clinical trial sites against a master CTMS site database.

For each row in the uploaded file, determine whether it refers to the same real-world clinical site as any entry in the CTMS database.
Use semantic matching — consider site name variations, abbreviations, alternate spellings, country codes vs. full names, and PI name formatting differences.
A match requires confident identification of the same physical site; do not match on superficial similarity alone.

Return a JSON object:
{
  "matches": [
    {
      "uploaded_index": <int, 0-based row index in the uploaded file>,
      "uploaded_identifier": "<best identifying string from the uploaded row>",
      "ctms_site_id": "<site_id from CTMS, e.g. SP-001>",
      "ctms_site_name": "<site_name from CTMS>",
      "match_confidence": "high|medium|low",
      "match_basis": "<brief explanation of why this is a match>"
    }
  ],
  "unmatched_indices": [<list of 0-based row indices with no match>],
  "summary": {
    "total_uploaded": <int>,
    "matched": <int>,
    "unmatched": <int>,
    "notes": "<any overall observations about match quality or ambiguities>"
  }
}

Rules:
- Only include a row in "matches" if you are reasonably confident it corresponds to a CTMS site.
- Each uploaded row may match at most one CTMS site.
- Each CTMS site may match at most one uploaded row.
- All uploaded row indices not in "matches" must appear in "unmatched_indices".
- Return ONLY the JSON object, no markdown fences, no other text."""

_SITE_MATCHING_USER = """Uploaded site list ({n_uploaded} rows, CSV with index):
{uploaded_data}

CTMS master site database (CSV):
{ctms_data}

Match each uploaded row to the most appropriate CTMS site, or mark it as unmatched."""

# -- Find CTMS file (try project root then workdir) ---------------------------
def _find_ctms():
    candidates = [
        _Path(_os.getcwd()) / "data" / "CTMS_SITES.csv",
        _Path(backend.__file__).parent.parent / "data" / "CTMS_SITES.csv",
        _Path(backend.__file__).parent.parent.parent / "data" / "CTMS_SITES.csv",
    ]
    for p in candidates:
        if p.exists():
            print(f"CTMS file found: {p}")
            return p
    print(f"CTMS not found. Tried: {[str(p) for p in candidates]}")
    return None

# -- Inline SiteListMatchingAgent ---------------------------------------------
class _SiteListMatchingAgent:
    skill_id    = "site_list_matching"
    display_name = "Clinical Site List Matching"

    def __init__(self, llm):
        self.llm = llm

    def run(self, params, state):
        from backend.agents.base_agent import AgentResult
        MAX_ROWS = 200

        file_info = state.uploaded_files.get("site_file")
        if not file_info:
            return AgentResult(success=False, text_response="",
                               error_message="Missing uploaded file: Site List File.")

        ctms_path = _find_ctms()
        if not ctms_path:
            return AgentResult(success=False, text_response="",
                               error_message="CTMS_SITES.csv not found.")
        ctms_df = _pd.read_csv(ctms_path)
        ctms_cols = [c for c in ["site_id","site_name","country","city","pi_name"]
                     if c in ctms_df.columns]
        ctms_text = ctms_df[ctms_cols].to_csv(index=False)

        uploaded_df = _pd.DataFrame(file_info["data"])
        n_uploaded = len(uploaded_df)
        truncated = uploaded_df.head(MAX_ROWS).to_csv(index=True)
        if n_uploaded > MAX_ROWS:
            truncated += f"
[Truncated to {MAX_ROWS} rows]"

        messages = [
            {"role": "system", "content": _SITE_MATCHING_SYSTEM},
            {"role": "user",   "content": _SITE_MATCHING_USER.format(
                uploaded_data=truncated,
                ctms_data=ctms_text,
                n_uploaded=min(n_uploaded, MAX_ROWS),
            )},
        ]
        try:
            raw = self.llm.complete_json(messages, temperature=self.llm.temp_deterministic)
            matches         = raw.get("matches", [])
            summary         = raw.get("summary", {})
        except Exception as e:
            return AgentResult(success=False, text_response="",
                               error_message=f"Error during site matching: {e}")

        n_matched   = summary.get("matched",   len(matches))
        n_unmatched = summary.get("unmatched", n_uploaded - n_matched)
        match_rate  = round(n_matched / max(n_uploaded, 1) * 100, 1)

        summary_text = (
            f"**Site List Matching Results**\n\n"
            f"Uploaded **{n_uploaded}** sites — compared against **{len(ctms_df)}** CTMS sites.\n\n"
            f"- Matched: **{n_matched}** ({match_rate}%)\n"
            f"- Unmatched: **{n_unmatched}**\n\n"
            f"{summary.get('notes','')}".strip()
        )

        matched_by_idx = {m["uploaded_index"]: m for m in matches}
        table_data = []
        for i in range(min(n_uploaded, MAX_ROWS)):
            row   = uploaded_df.iloc[i]
            ident = str(next((row.get(k) for k in
                              ["site_name","name","Site Name","Site"] if row.get(k)), row.iloc[0]))
            m = matched_by_idx.get(i)
            table_data.append({
                "Row":           i + 1,
                "Uploaded Site": ident,
                "Match Status":  "Matched" if m else "Not matched",
                "CTMS Site ID":  m.get("ctms_site_id","")   if m else "",
                "CTMS Site Name":m.get("ctms_site_name","") if m else "",
                "Confidence":    m.get("match_confidence","") if m else "",
                "Match Basis":   m.get("match_basis","")    if m else "",
            })

        return AgentResult(
            success=True,
            text_response=summary_text,
            table_data=table_data,
            table_columns=["Row","Uploaded Site","Match Status",
                           "CTMS Site ID","CTMS Site Name","Confidence","Match Basis"],
        )

# -- Register inline agent under both IDs (handles any workdir state) ---------
_matching_agent = _SiteListMatchingAgent(orchestrator.llm)
orchestrator.router._registry["site_list_matching"] = _matching_agent
orchestrator.router._registry["site_list_merger"]   = _matching_agent

# -- Schema alias: whichever key exists, ensure both exist --------------------
_schema = (orchestrator.schemas.get("site_list_matching")
           or orchestrator.schemas.get("site_list_merger"))
if _schema:
    orchestrator.schemas["site_list_matching"] = _schema
    orchestrator.schemas["site_list_merger"]   = _schema

print(f"Router skills:  {list(orchestrator.router._registry.keys())}")
print(f"Schema skills:  {list(orchestrator.schemas.keys())}")

# ── Reload schemas from project root (bypasses Dataiku workdir YAML cache) ──
from backend.state.parameter_schema import load_schemas
import os as _os
_skills_cfg = Path(_os.getcwd()) / "config" / "skills_config.yaml"
if not _skills_cfg.exists():
    # fallback: path relative to backend package location
    _skills_cfg = Path(backend.__file__).parent.parent / "config" / "skills_config.yaml"
orchestrator.schemas = load_schemas(str(_skills_cfg))
print(f"Schemas loaded from: {_skills_cfg}")
print(f"Skills available: {list(orchestrator.schemas.keys())}")

# ── LLM connectivity test ────────────────────────────────────────────────────
print(f"LLM connection ID: {orchestrator.llm.connection_id}")
try:
    _test = orchestrator.llm.complete(
        [{"role": "user", "content": "Reply with the single word: OK"}],
    )
    print(f"LLM connectivity test PASSED — response: {_test[:80]!r}")
except Exception as _e:
    print(f"LLM connectivity test FAILED: {_e}")


In [ ]:
# Cell 4 — Build and launch the interactive UI
import uuid
import pandas as pd
import panel as pn
from panel.chat import ChatInterface
from backend.state.conversation_state import FSMState
from backend.agents.site_list_merger_agent import parse_uploaded_file

SESSION_ID = str(uuid.uuid4())

# ── Widen chat message bubbles (override Panel defaults) ──────────────────────
pn.config.raw_css.append("""
/* Stretch message rows and bubbles to full feed width */
.chat-feed-log .chat-message-row { max-width: 100% !important; width: 100% !important; }
.chat-feed-log .chat-entry       { max-width: 100% !important; width: 100% !important; }
.chat-message                    { max-width: 100% !important; width: 100% !important; }
/* Remove any fixed pixel width on the message content pane */
.chat-message .pn-panel          { max-width: 100% !important; width: 100% !important; }
.bk-panel-models-chat-ChatMessage { max-width: 100% !important; }
""")


# ── LLM trace log ────────────────────────────────────────────────────────────
_trace_content = pn.pane.Markdown(
    '*No LLM calls yet — send a message to see the trace.*',
    sizing_mode='stretch_width',
    styles={'font-size': '11px', 'white-space': 'pre-wrap', 'font-family': 'monospace', 'color': '#222'},
)


def _infer_call_label(messages):
    system = next((m["content"] for m in messages if m["role"] == "system"), "").lower()
    if system.startswith("[citeline semantic mapping]"):  return "Citeline Semantic Mapping"
    if system.startswith("[citeline filter result]"):     return "Citeline Filter Result"
    if "routing assistant" in system or ("intent" in system and "skill" in system):
        return "Intent Classification"
    if "parameter extraction" in system: return "Parameter Extraction"
    if "site list" in system or "ctms" in system: return "Site List Matching Agent"
    if "benchmarking" in system or "benchmark" in system: return "Trial Benchmarking Agent"
    if "reimbursement" in system or "hta" in system: return "Drug Reimbursement Agent"
    if "enrollment" in system and ("pessimistic" in system or "scenario" in system):
        return "Enrollment Params Estimation"
    if "narrative" in system or ("enrollment" in system and "interpret" in system):
        return "Enrollment Narrative"
    if system.startswith("["):  return system.strip("[]").title()
    return "LLM Call"

def _update_trace_log():
    log = getattr(orchestrator.llm, "call_log", [])
    conn_id = getattr(orchestrator.llm, "connection_id", "unknown")
    if not log:
        _trace_content.object = (
            f"Connection ID in use: {conn_id}\n\n"
            "No LLM calls recorded yet.\n"
            "If this stays empty after sending a message, the LLM call is failing\n"
            "before it can be logged — check Cell 3 output for import errors."
        )
        return
    entries = []
    for i, entry in enumerate(log, 1):
        msgs      = entry.get("messages", [])
        resp_text = entry.get("response", "")
        is_error  = entry.get("error", False)
        is_synth  = entry.get("synthetic", False)
        label     = _infer_call_label(msgs)
        if is_synth:
            lines = [f"=== {i}: {label} ==="]
            lines.append(resp_text)
        elif is_error:
            lines = [f"!!! ERROR Call {i}: {label} (conn={conn_id}) !!!"]
            for msg in msgs:
                role = msg.get("role", "user").upper()
                content = msg.get("content", "")[:600]
                lines.append(f"[{role}]\n{content}")
            lines.append(f"[RESPONSE]\n{resp_text[:800]}")
        else:
            lines = [f"--- Call {i}: {label} (conn={conn_id}) ---"]
            for msg in msgs:
                role = msg.get("role", "user").upper()
                content = msg.get("content", "")[:600]
                content += ("...[truncated]" if len(msg.get("content","")) > 600 else "")
                lines.append(f"[{role}]\n{content}")
            resp_trunc = resp_text[:800] + ("...[truncated]" if len(resp_text) > 800 else "")
            lines.append(f"[RESPONSE]\n{resp_trunc}")
        entries.append("\n".join(lines))
    _trace_content.object = "\n\n".join(entries)


# ── Response → Panel ─────────────────────────────────────────────────────────
def _response_to_panel(resp):
    items = []

    if resp.get('message'):
        items.append(pn.pane.Markdown(resp['message'], sizing_mode='stretch_width'))

    if resp.get('table_data'):
        rows    = resp['table_data']
        columns = resp.get('table_columns')
        df = pd.DataFrame(rows, columns=columns) if columns else pd.DataFrame(rows)
        items.append(pn.pane.DataFrame(
            df, sizing_mode='stretch_width', max_rows=200,
            styles={'font-size': '12px'},
        ))

    if resp.get('chart_json'):
        _chart = resp['chart_json']
        if isinstance(_chart, dict):
            items.append(pn.pane.Markdown('*Chart unavailable.*'))
        else:
            items.append(pn.pane.Bokeh(_chart, sizing_mode='stretch_width'))

    if resp.get('fsm_state') == 'confirmation_pending':
        yes_btn = pn.widgets.Button(
            name='\u2713  Yes, proceed', button_type='success',
            width=160, margin=(8, 6, 4, 0),
        )
        no_btn = pn.widgets.Button(
            name='\u2717  Cancel', button_type='danger',
            width=110, margin=(8, 0, 4, 0),
        )

        def _confirm(event):
            yes_btn.disabled = True
            no_btn.disabled = True
            r = orchestrator.handle_confirmation(SESSION_ID, confirmed=True)
            chat.send(_response_to_panel(r), user='Assistant', respond=False)
            _maybe_show_export(r)

        def _cancel(event):
            yes_btn.disabled = True
            no_btn.disabled = True
            r = orchestrator.handle_confirmation(SESSION_ID, confirmed=False)
            chat.send(_response_to_panel(r), user='Assistant', respond=False)

        yes_btn.on_click(_confirm)
        no_btn.on_click(_cancel)
        items.append(pn.Row(yes_btn, no_btn, margin=(4, 0, 0, 0)))

    if resp.get('result_id'):
        _maybe_show_export(resp)

    return pn.Column(*items, sizing_mode='stretch_width') if items else pn.pane.Markdown('...')


# ── Export bar ───────────────────────────────────────────────────────────────
_current_result_id = None

export_input  = pn.widgets.TextInput(placeholder='Dataiku dataset name\u2026', width=260)
export_button = pn.widgets.Button(name='\u2b07  Export to Dataiku Dataset',
                                   button_type='primary', width=220)
export_status = pn.pane.Markdown('', width=400)
export_row = pn.Column(
    pn.pane.Markdown('**Export last result:**', margin=(0, 0, 4, 0)),
    pn.Row(export_input, export_button, export_status),
    visible=False,
    styles={'background': '#f0f4ff', 'padding': '8px', 'border-radius': '6px'},
)

def _maybe_show_export(resp):
    global _current_result_id
    if resp.get('result_id'):
        _current_result_id = resp['result_id']
        export_row.visible = True

def _on_export(event):
    dataset_name = export_input.value.strip()
    if not dataset_name:
        export_status.object = '\u26a0 Enter a dataset name first.'
        return
    if not _current_result_id:
        export_status.object = '\u26a0 No result to export.'
        return
    r = orchestrator.export_to_dataset(SESSION_ID, _current_result_id, dataset_name)
    export_status.object = r.get('message', 'Done.')

export_button.on_click(_on_export)


# ── CRO file upload (Site List Merger) ───────────────────────────────────────
upload_cro    = pn.widgets.FileInput(accept='.csv,.xlsx,.xls', width=260)
upload_status = pn.pane.Markdown(
    '', sizing_mode='stretch_width',
    styles={'font-size': '12px', 'color': '#444'},
)

class _FakeFileStorage:
    def __init__(self, filename, data):
        self.filename = filename
        self._data = data
    def read(self):
        return self._data

def _on_cro_upload(event):
    if upload_cro.value is None:
        return
    filename = upload_cro.filename or 'upload'
    try:
        parsed = parse_uploaded_file(_FakeFileStorage(filename, upload_cro.value))
        state = orchestrator.session_store.get_or_create(SESSION_ID)
        state.uploaded_files['site_file'] = parsed
        state.active_skill = 'site_list_matching'
        state.fsm_state = FSMState.PARAMETER_GATHERING
        upload_status.object = (
            f'\u2705 **CRO file loaded:** {filename} — {len(parsed["data"])} rows, '
            f'columns: {", ".join(parsed["columns"])}'
        )
        chat.send(
            f'Site list uploaded: **{filename}** ({len(parsed["data"])} rows). '
            f'Type "match" or "proceed" to run matching against the CTMS database.',
            user='Assistant', respond=False,
        )
    except ValueError as e:
        upload_status.object = f'\u26a0 **Upload error:** {e}'

upload_cro.param.watch(_on_cro_upload, 'value')

upload_bar = pn.Row(
    pn.pane.Markdown(
        '\U0001f4c2 **CRO Site List** *(for Site List Merger)*',
        margin=(6, 10, 0, 0),
        styles={'font-size': '13px', 'white-space': 'nowrap'},
    ),
    upload_cro,
    upload_status,
    sizing_mode='stretch_width',
    styles={
        'background': '#f9f9f9',
        'padding': '8px 12px',
        'border-bottom': '1px solid #e0e0e0',
    },
    align='center',
)


# ── Chat callback ─────────────────────────────────────────────────────────────
def chat_callback(contents, user, instance):
    try:
        resp = orchestrator.process_message(SESSION_ID, contents)
    except Exception as _exc:
        import traceback
        tb = traceback.format_exc()
        _trace_content.object = f"EXCEPTION in process_message:\n{tb}"
        return pn.pane.Markdown(f"**Error:** `{_exc}`  \nSee trace pane for full traceback.")
    _maybe_show_export(resp)
    _update_trace_log()
    return _response_to_panel(resp)


chat = ChatInterface(
    callback=chat_callback,
    user='You',
    avatar='\U0001f464',
    callback_user='Assistant',
    show_rerun=False,
    show_undo=False,
    show_copy_icon=False,
    placeholder_text='Describe what you need\u2026',
    sizing_mode='stretch_width',
    max_width=None,
    height=720,
    message_params={"sizing_mode": "stretch_width", "max_width": None},
    styles={'border': '1px solid #e0e0e0', 'border-radius': '8px'},
)

chat.send(
    pn.pane.Markdown(
        "Hello! I'm your **Clinical Analytics Assistant**. I can help you with:\n\n"
        "1. **Site List Matching** \u2014 Upload a site list and match it against the CTMS master database  \n"
        "2. **Trial Benchmarking** \u2014 Benchmark trials by indication, age group, and phase  \n"
        "3. **Drug Reimbursement** \u2014 Assess reimbursement outlook by country  \n"
        "4. **Enrollment Forecasting** \u2014 Generate pessimistic / moderate / optimistic enrollment curves  \n\n"
        "What would you like to do?"
    ),
    user='Assistant', respond=False,
)


# ── App layout ────────────────────────────────────────────────────────────────
app = pn.Column(
    pn.pane.Markdown(
        '# Clinical Analytics Assistant\n*Drug R&D | Dataiku Notebook*',
        styles={'background': '#1a1a2e', 'color': 'white',
                'padding': '14px 18px', 'border-radius': '10px 10px 0 0',
                'margin': '0'},
    ),
    upload_bar,
    chat,
    export_row,
    pn.Column(
        pn.pane.Markdown('### LLM Call Trace', margin=(0, 0, 4, 0),
                       styles={'color': '#222'}),
        _trace_content,
        sizing_mode='stretch_width',
        height=320,
        scroll=True,
        styles={'background': '#f7f7f7', 'padding': '10px',
                'border': '1px solid #ddd', 'border-radius': '6px'},
    ),
    sizing_mode='stretch_width',
    styles={'border': '1px solid #ccc', 'border-radius': '10px',
            'overflow': 'hidden'},
)

app.servable()
app